# Preprocessing dan Training Model 2 (Crowd Classifier)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("../data/crowd_dataset.csv")
print(df.shape)
print(df.info())
print(df.isna().sum())

(17526, 8)
<class 'pandas.DataFrame'>
RangeIndex: 17526 entries, 0 to 17525
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   stop_id      17526 non-null  int64  
 1   day_of_week  17526 non-null  int64  
 2   hour         17526 non-null  int64  
 3   is_holiday   17526 non-null  int64  
 4   weather      17526 non-null  int64  
 5   tap_count    17526 non-null  int64  
 6   prev_count   17526 non-null  float64
 7   crowd_level  17526 non-null  str    
dtypes: float64(1), int64(6), str(1)
memory usage: 1.1 MB
None
stop_id        0
day_of_week    0
hour           0
is_holiday     0
weather        0
tap_count      0
prev_count     0
crowd_level    0
dtype: int64


In [ ]:
X = df.drop(["tap_count", "crowd_level"], axis=1)
y = df["crowd_level"]

target_mapping = {
    'Sepi': 0,
    'Normal': 1,
    'Padat': 2,
    'Penuh': 3
}

y_encoded = y.map(target_mapping)
print(y_encoded.isna().sum())

0


Definisikan fitur untuk proses training dimana kolom tap_count dan crowd_level dari kolom kolom yang dijadikan sebagai fitur karena tap_count merupakan sumber label crowd level membiarkannya masuk sebagai fitur berpotensi data leakage, sedangkan crowd level merupakan target yang harus dipisah. Kemudian target di-encode dengan cara mapping ke skala ordinal 0-3 berdasarkan urutan level yang benar bukan secara alfabetis seperti yang digunakan LabelEncoder. Encoding berhasil dengan seluruh label berhasil ter-encode tanpa ada data yang kosong.

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(14020, 6) (3506, 6)
crowd_level
1    0.407275
0    0.384593
2    0.156633
3    0.051498
Name: proportion, dtype: float64
crowd_level
1    0.407302
0    0.384484
2    0.156589
3    0.051626
Name: proportion, dtype: float64


Pisahkan data training dan testing dengan stratified split berdasarkan y yang wajib karena Penuh hanya 5.1% dan kalau tidak stratified bisa kosong di test set. Dihasilkan pemisahan yang sesuai dengan distribusi asli (Sepi ~38.5%, Normal ~40.7%, Padat ~15.6%, Penuh ~5.15% dan sudah konsisten di kedua split)

## Training

Model yang digunakan dalam proses training ini adalah Gradient Boosting Classifier sebagai model utama dan Random Forest Classifier sebagai baseline pembanding. Proses training dimulai dengan parameter default dahulu tanpa weighting, untuk melihat seberapa parah Penuh dirugikan secara natural, baru putuskan perlu weighting atau tidak

In [9]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

for name, y_pred in [("GB", y_pred_gb), ("RF", y_pred_rf)]:
    print(name, "accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))

GB accuracy: 0.8633770678836281
              precision    recall  f1-score   support

           0       0.94      0.95      0.94      1348
           1       0.89      0.87      0.88      1428
           2       0.67      0.80      0.73       549
           3       0.74      0.37      0.49       181

    accuracy                           0.86      3506
   macro avg       0.81      0.75      0.76      3506
weighted avg       0.87      0.86      0.86      3506

[[1277   71    0    0]
 [  78 1242  108    0]
 [   0   84  441   24]
 [   0    0  114   67]]
RF accuracy: 0.8648031945236737
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      1348
           1       0.87      0.89      0.88      1428
           2       0.71      0.74      0.72       549
           3       0.71      0.51      0.59       181

    accuracy                           0.86      3506
   macro avg       0.81      0.77      0.78      3506
weighted avg       0.86     

Hasil training tersebut telah melampaui target dengan akurasi GB 0.863 dan RF 0.865, jauh di atas threshold target yakni >0.70. Dari sisi target, ini sudah aman tapi confusion matrix mengonfirmasi hipotesis yang dibuat saat EDA, antara lain:

- Penuh (kelas 3) lemah di kedua model dengan GB recall 0.37, RF recall 0.51, RF lebih unggul
- Mayoritas error Penuh jatuh ke Padat, persis seperti yang diprediksi dari overlap boxplot: GB salah klasifikasi 114/181 (63%) Penuh sebagai Padat. RF sedikit lebih baik (89/181 = 49%).
- Precision Penuh GB cukup tinggi (0.74). Masalahnya bukan precision, tapi GB jarang memprediksi Penuh sama sekali (under-trigger ke kelas minoritas).
- Sepi dan Normal (kelas 0 dan 1) terpisah bersih di kedua model sesuai hipotesis EDA

Untuk saat ini belum waktunya putuskan sample_weight. Recall Penuh GB (0.37) masih di atas ambang "sangat buruk" (<30%) yang ditetapkan, jadi belum otomatis perlu weighting. Selanjutnya, cek feature importance terlebih dulu untuk konfirmasi hipotesis saat EDA sebelumnya

In [10]:
import pandas as pd
importances = pd.Series(gb.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print(importances)

hour           0.503985
is_holiday     0.244276
day_of_week    0.195039
weather        0.049160
prev_count     0.004843
stop_id        0.002698
dtype: float64


Hasilnya hour dominan (50.4%) yang mengonfirmasi hipotesis EDA yang benar. Korelasi Pearson rendah (-0.09) ternyata memang bukan representasi kontribusi riil, feature importance tree-based menunjukkan hour justru fitur paling informatif. Tapi ada dua hal yang berlawanan dari ekspektasi awal, antara lain:

- prev_count mendekati nol (0.0048) yang justru kebalikan dari yang dikhawatirkan saat EDA. Faktanya model hampir tidak memakainya sama sekali. Ini justru mendukung kecurigaan saat EDA noise dari lag posisi di dataset bukan yang ketat per jam membuat fitur ini hampir tidak berguna buat split tree

- stop_id mendekati nol (0.0027), ini yang lebih perlu dipertanyakan. Label crowd_level di-derive dari load_pct yang relatif terhadap kapasitas tiap halte (269 halte, 3 tier kapasitas). Kalau kapasitas benar-benar berpengaruh ke threshold Padat/Penuh, stop_id seharusnya membawa sinyal tier itu tapi importance-nya hampir nol

Selanjutnya ablation test dengan menghilangkan satu variabel dulu. Kalau drop prev_count tidak menurunkan (atau malah menaikkan) recall Penuh, itu bukti kuat untuk drop permanen fitur ini

In [11]:
X_train_noprev = X_train.drop("prev_count", axis=1)
X_test_noprev = X_test.drop("prev_count", axis=1)

gb2 = GradientBoostingClassifier(random_state=42)
gb2.fit(X_train_noprev, y_train)
y_pred_gb2 = gb2.predict(X_test_noprev)
print(classification_report(y_test, y_pred_gb2))

              precision    recall  f1-score   support

           0       0.94      0.95      0.94      1348
           1       0.89      0.87      0.88      1428
           2       0.65      0.79      0.72       549
           3       0.68      0.33      0.44       181

    accuracy                           0.86      3506
   macro avg       0.79      0.73      0.74      3506
weighted avg       0.86      0.86      0.86      3506



Hasilnya menurun, dan ini hasil yang penting karena hipotesis "prev_count cuma noise" terbantahkan oleh data, bukan oleh importance score. Recall Penuh turun dari 0.37 ke 0.33, precision 0.74 ke 0.68, recall Padat juga turun sedikit (0.80 ke 0.79). Importance rendah (0.0048) ternyata tidak berarti kontribusinya nol dan ada kemungkinan dia membantu di kasus kasus dekat boundary yang justru paling sulit (antara Padat dan Penuh). Kesimpulannya kembalikan prev_count ke fitur, jangan drop. Selanjutnya yang akan dicoba adalah sample_weight manual, kalau membaik tanpa accuracy overall turun drastis bisa jadi kandidat model final. Kalau malah memperburuk precision Penuh sampai tidak masuk akal (banyak false positive Penuh), berarti weighting terlalu agresif dan perlu pertimbangkan versi lebih moderat atau terima recall 

In [12]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

gb3 = GradientBoostingClassifier(random_state=42)
gb3.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_gb3 = gb3.predict(X_test)
print(classification_report(y_test, y_pred_gb3))

              precision    recall  f1-score   support

           0       0.94      0.95      0.95      1348
           1       0.91      0.84      0.87      1428
           2       0.67      0.71      0.69       549
           3       0.50      0.70      0.58       181

    accuracy                           0.85      3506
   macro avg       0.76      0.80      0.77      3506
weighted avg       0.86      0.85      0.86      3506



In [13]:
print(confusion_matrix(y_test, y_pred_gb3))

[[1279   69    0    0]
 [  78 1198  139   13]
 [   0   49  388  112]
 [   0    0   55  126]]


Yang sebenarnya terjadi disini bisa dibilang hanya trade-off. Model jadi lebih agresif memprediksi Penuh (recall naik) tapi separuh prediksi Penuh sekarang salah (precision 0.50), dan korbannya recall Normal & Padat ikut turun, serta Macro F1 nyaris tidak berubah (0.76 ke 0.77). Konfirmasi dari confusion matrix:

- Spillover Padat ke Penuh memang melonjak seperti diduga. Sebelumnya 24 baris Padat salah ke Penuh, sekarang 112 (naik 4.7x). Ini efek langsung dari weighting yang mendorong batas keputusan terlalu jauh ke arah kelas minoritas.

- Ada temuan baru yang lebih perlu diwaspadai, yakni muncul 13 baris Normal yang loncat langsung ke Penuh (sebelumnya 0). Ini bukan confusion antarkelas yang bertetangga seperti Padat dan Penuh yang sudah diprediksi dari EDA melainkan ini lompatan dua tingkat severity (Normal langsung dianggap Penuh). Errornya kecil secara volume (13/1428 = 0.9%), tapi secara konsep ini failure mode baru yang tidak ada di model sebelumnya.

Eksperimen berikutnya dibuat batch independen satu sama lain:

- Moderate weighting dengan sqrt(balanced_weight) 
- RF dengan class_weight='balanced' native
- LightGBM dengan class_weight='balanced', belum pernah dicoba untuk Model 2

In [ ]:
balanced_w = compute_sample_weight(class_weight='balanced', y=y_train)
moderate_w = np.sqrt(balanced_w)

# GB 
gb4 = GradientBoostingClassifier(random_state=42)
gb4.fit(X_train, y_train, sample_weight=moderate_w)
y_pred_gb4 = gb4.predict(X_test)
print("GB moderate weight"); print(classification_report(y_test, y_pred_gb4)); print(confusion_matrix(y_test, y_pred_gb4))

# RF 
rf2 = RandomForestClassifier(random_state=42, class_weight='balanced')
rf2.fit(X_train, y_train)
y_pred_rf2 = rf2.predict(X_test)
print("RF balanced"); print(classification_report(y_test, y_pred_rf2)); print(confusion_matrix(y_test, y_pred_rf2))

# LightGBM 
import lightgbm as lgb
lgbm = lgb.LGBMClassifier(random_state=42, class_weight='balanced')
lgbm.fit(X_train, y_train)
y_pred_lgbm = lgbm.predict(X_test)
print("LightGBM balanced"); print(classification_report(y_test, y_pred_lgbm)); print(confusion_matrix(y_test, y_pred_lgbm))

GB moderate weight
              precision    recall  f1-score   support

           0       0.94      0.95      0.94      1348
           1       0.90      0.85      0.87      1428
           2       0.67      0.77      0.71       549
           3       0.56      0.56      0.56       181

    accuracy                           0.86      3506
   macro avg       0.77      0.78      0.77      3506
weighted avg       0.86      0.86      0.86      3506

[[1278   70    0    0]
 [  79 1209  132    8]
 [   0   57  420   72]
 [   0    0   79  102]]
RF balanced
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      1348
           1       0.89      0.86      0.88      1428
           2       0.68      0.78      0.73       549
           3       0.64      0.54      0.59       181

    accuracy                           0.86      3506
   macro avg       0.79      0.78      0.78      3506
weighted avg       0.87      0.86      0.86      3506

[[1272

Temuan penting dari hasil eksperimen ini dan perbandingannya dengan eksperimen sebelumnya adalah sebagai berikut:
- GB moderate weight berhasil menekan agresivitas dari percobaan sebelumnya dengan leak Penuh ke Padat turun ke 72 dan precision naik ke 0.0.56, namun recall Penuh ikut terkompresi menjadi 0.56 yang mengindikasikan bahwa pendekatan sqrt belum menemukan titik optimal antara sensitivitas dan presisi

- RF balanced mencapai F1 Penuh yang setara dengan GB moderate (0.59) dengan leak Padat ke Penuh yang lebih rendah (50) dan precision lebih baik (0.64), namun recall-nya stagnan di 0.54, artinya hampir separuh kasus Penuh masih terlewat. 

- LightGBM balanced menjadi kandidat terkuat dengan recall Penuh mencapai 0.70 yang tertinggi di antara model lain dalam eksperimen ini dan setara dengan GB full balanced yang jauh lebih agresif, menunjukkan bahwa LightGBM mampu mengekstrak sensitivitas yang sama dengan GB full balanced tanpa biaya presisi dan leak yang sama besarnya



Eksperimen selanjutnya akan menguji anomali baru yang ditemukan sebelumnya, apakah rendahnya feature importance stop_id pada model sebelumnya disebabkan oleh tree yang terlalu dangkal, atau memang fitur tersebut lemah secara inheren. Logikanya pola yang melibatkan stop_id kemungkinan baru muncul pada interaksi fitur yang lebih dalam, sehingga tree yang terlalu shallow tidak akan pernah menjangkaunya. Dilakukan hyperparameter tuning dengan Grid search pada max_depth dan n_estimators dijalankan hanya pada dua kandidat finalis (GB moderate dan LightGBM balanced) yang terbukti kompetitif untuk menjawab hipotesis ini secara efisien tanpa mengulang semua model.

In [15]:
from sklearn.model_selection import ParameterGrid

param_grid = {'max_depth': [3, 5, 7], 'n_estimators': [100, 200]}

for params in ParameterGrid(param_grid):
    gb_tune = GradientBoostingClassifier(random_state=42, **params)
    gb_tune.fit(X_train, y_train, sample_weight=moderate_w)
    pred = gb_tune.predict(X_test)
    print(params, "recall_penuh:", classification_report(y_test, pred, output_dict=True)['3']['recall'],
          "acc:", accuracy_score(y_test, pred))

for params in ParameterGrid(param_grid):
    lgbm_tune = lgb.LGBMClassifier(random_state=42, class_weight='balanced', **params, verbose=-1)
    lgbm_tune.fit(X_train, y_train)
    pred = lgbm_tune.predict(X_test)
    print(params, "recall_penuh:", classification_report(y_test, pred, output_dict=True)['3']['recall'],
          "acc:", accuracy_score(y_test, pred))

{'max_depth': 3, 'n_estimators': 100} recall_penuh: 0.56353591160221 acc: 0.8582430119794637
{'max_depth': 3, 'n_estimators': 200} recall_penuh: 0.580110497237569 acc: 0.8628066172276098
{'max_depth': 5, 'n_estimators': 100} recall_penuh: 0.6022099447513812 acc: 0.8650884198516828
{'max_depth': 5, 'n_estimators': 200} recall_penuh: 0.6077348066298343 acc: 0.8639475185396464
{'max_depth': 7, 'n_estimators': 100} recall_penuh: 0.6077348066298343 acc: 0.869937250427838
{'max_depth': 7, 'n_estimators': 200} recall_penuh: 0.6022099447513812 acc: 0.8667997718197376
{'max_depth': 3, 'n_estimators': 100} recall_penuh: 0.6795580110497238 acc: 0.8513976041072447
{'max_depth': 3, 'n_estimators': 200} recall_penuh: 0.7182320441988951 acc: 0.8482601254991443
{'max_depth': 5, 'n_estimators': 100} recall_penuh: 0.7182320441988951 acc: 0.8494010268111808
{'max_depth': 5, 'n_estimators': 200} recall_penuh: 0.6629834254143646 acc: 0.8505419281232174
{'max_depth': 7, 'n_estimators': 100} recall_penuh: 0.

Eksperimen grid search mengkonfirmasi hipotesis stop_id dimana fitur tersebut tidak lemah secara inheren, melainkan membutuhkan tree yang cukup dalam untuk mengeksploitasi 269 kategorinya. Pada GB, pendalaman tree dari default depth=3 secara konsisten memperbaiki recall dan accuracy sekaligus namun dengan depth=7, n=100 sebagai titik optimal, recall Penuh 0.608 dan accuracy 0.870, yang merupakan accuracy tertinggi dari seluruh varian GB yang pernah diuji. LightGBM menunjukkan dinamika berbeda karena default-nya sudah unlimited depth, sehingga pembatasan ke depth=3 justru menurunkan performa, namun depth=5, n=100 menemukan titik yang lebih baik dari konfigurasi default sebelumnya, dengan recall naik dari 0.70 ke 0.718 meski accuracy turun tipis dari 0.856 ke 0.849. Dua finalis yang terbentuk mencerminkan trade-off yang jelas: GB depth=7 unggul di accuracy (0.870 vs 0.849) sementara LightGBM depth=5 unggul di recall Penuh (0.718 vs 0.608). Masih perlu report lengkap untuk menentukan pemenangnya.

In [9]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import lightgbm as lgb

balanced_w = compute_sample_weight(class_weight='balanced', y=y_train)
moderate_w = np.sqrt(balanced_w)

gb_final = GradientBoostingClassifier(random_state=42, max_depth=7, n_estimators=100)
gb_final.fit(X_train, y_train, sample_weight=moderate_w)
pred_gb_final = gb_final.predict(X_test)
print("GB final candidate")
print(classification_report(y_test, pred_gb_final))
print(confusion_matrix(y_test, pred_gb_final))

lgbm_final = lgb.LGBMClassifier(random_state=42, max_depth=5, n_estimators=100, class_weight='balanced', verbose=-1)
lgbm_final.fit(X_train, y_train)
pred_lgbm_final = lgbm_final.predict(X_test)
print("LightGBM final candidate")
print(classification_report(y_test, pred_lgbm_final))
print(confusion_matrix(y_test, pred_lgbm_final))

GB final candidate
              precision    recall  f1-score   support

           0       0.95      0.94      0.94      1348
           1       0.89      0.87      0.88      1428
           2       0.70      0.79      0.74       549
           3       0.71      0.61      0.65       181

    accuracy                           0.87      3506
   macro avg       0.81      0.80      0.80      3506
weighted avg       0.87      0.87      0.87      3506

[[1270   78    0    0]
 [  73 1238  112    5]
 [   0   77  432   40]
 [   0    0   71  110]]
LightGBM final candidate
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      1348
           1       0.90      0.83      0.87      1428
           2       0.66      0.71      0.68       549
           3       0.51      0.72      0.59       181

    accuracy                           0.85      3506
   macro avg       0.75      0.80      0.77      3506
weighted avg       0.86      0.85      0.85     

GB final (depth=7, moderate weight) menjadi model terpilih setelah perbandingan full classification report melawan LightGBM final (depth=5, balanced). Meskipun LightGBM unggul di recall mentah Penuh (0.72 vs 0.61), keunggulan tersebut dibayar mahal dengan precision Penuh anjlok ke 0.51 dan leak Padat ke Penuh melonjak ke 115 yang hampir tiga kali lipat GB final (40), sehingga F1 Penuh LightGBM kalah (0.59 vs 0.65).

GB final menang di accuracy (0.87), macro F1 (0.80), F1 Penuh, dan kedua jumlah leak metric sekaligus. Accuracy GB final (0.87) bahkan melampaui baseline awal (0.863), mengkonfirmasi bahwa depth tuning bukan sekadar memperbaiki imbalance handling, melainkan menghasilkan model yang secara genuine lebih baik secara keseluruhan. 

Selanjutnya dilakukan cross validation 5-fold dijalankan pada seluruh data (bukan X_train saja) menggunakan konfigurasi final GB (max_depth=7, n_estimators=100, moderate weight) untuk mendapatkan estimasi generalisasi yang lebih jujur, mengingat test set yang dipakai sepanjang eksperimen sebelumnya dalam praktiknya telah berfungsi sebagai validation set dan kemungkinan membuat angka performa sebelumnya sedikit optimis.

In [11]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
acc_scores, macro_f1_scores, recall_penuh_scores = [], [], []

for train_idx, val_idx in skf.split(X, y_encoded):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y_encoded.iloc[train_idx], y_encoded.iloc[val_idx]
    w_tr = np.sqrt(compute_sample_weight(class_weight='balanced', y=y_tr))

    model = GradientBoostingClassifier(random_state=42, max_depth=7, n_estimators=100)
    model.fit(X_tr, y_tr, sample_weight=w_tr)
    pred = model.predict(X_val)

    acc_scores.append(accuracy_score(y_val, pred))
    macro_f1_scores.append(f1_score(y_val, pred, average='macro'))
    recall_penuh_scores.append(recall_score(y_val, pred, labels=[3], average='macro'))

print(f"Accuracy: {np.mean(acc_scores):.4f} | {np.std(acc_scores):.4f}")
print(f"Macro F1: {np.mean(macro_f1_scores):.4f} | {np.std(macro_f1_scores):.4f}")
print(f"Recall Penuh: {np.mean(recall_penuh_scores):.4f} | {np.std(recall_penuh_scores):.4f}")

Accuracy: 0.8682 | 0.0045
Macro F1: 0.8108 | 0.0118
Recall Penuh: 0.6800 | 0.0435


Hasil cross validation (CV) 5-fold mengkonfirmasi bahwa test set yang telah digunakan berulang kali sepanjang eksperimen tidak menyebabkan overfitting yang signifikan. Accuracy 0.868 +- 0.005 menunjukkan stabilitas yang tinggi antar fold, sementara Macro F1 0.811 +- 0.012 melampaui hasil single-split (0.80), mengindikasikan bahwa performa model konsisten di berbagai partisi data. Recall Penuh 0.68 +- 0.044 lebih tinggi dari single-split (0.61), namun memiliki standar deviasi terbesar di antara ketiga metrik sebagai konsekuensi statistik yang wajar dari ukuran kelas Penuh yang kecil (~180 sampel per fold), sehingga recall kelas ini memiliki confidence interval yang lebih lebar dan tidak bisa diklaim sebagai angka tunggal yang pasti. Sebagai catatan limitasi: hyperparameter dipilih melalui eksplorasi single-split, dan CV final ini dijalankan sebagai estimasi generalisasi tambahan sehingga ada potensi optimisme residual kecil yang tidak sepenuhnya tereliminasi. Dengan accuracy CV 0.868 yang jauh melampaui target >0.70, Model 2 dinyatakan selesai dengan konfigurasi final GB max_depth=7, n_estimators=100, sqrt-balanced sample weight

In [12]:
import joblib

joblib.dump(gb_final, "model2_crowd_classifier.pkl")
joblib.dump(target_mapping, "model2_target_mapping.pkl")  
joblib.dump(list(X_train.columns), "model2_feature_order.pkl")  

['model2_feature_order.pkl']